In [ ]:
from platform import python_version
print(python_version())

### This notebook is an study over the paper "RNA velocity of single cells".


#### Theoretical description of RNA velocity

Based on the model of transcription shown in Fig. 1, we can write down the rate equations for asingle gene, which describes how the expected number of unspliced mRNA molecules 𝑢, andspliced molecules 𝑠, evolve over time:


$\frac{du}{dt} = \alpha(t) - \beta(t)u(t)$


$\frac{ds}{dt} = \beta(t)u(t) - \gamma(t)s(t)$



![solve_equation](../pictures/solve_equation.png)

Figure 1. Master equation and model predictions of different dynamics. a. A typical dynamics obtained by ourmodel of transcription as in Fig 1a, here we compute the full probability distribution over the counts described bythe master equation Ψ(s, u, t). b. A realization of the master equation and the expected value of the u-s relationshipshown as a phase portrait. Dots show individual samples (n=500) from the master equation shown in (a), sampleduniformly in time. Jitter was added for clarity, since samples are strictly integers. c. The effect of differenttranscription rates (α) on the u-s dynamics. d. Dynamics corresponding to different degradation rates. e. Left, thesolution of the rate equations for an oscillating transcription rate (e.g. like in a biological clock). Right, the phaseportrait of the same solution with a realization of Ψ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.integrate import solve_ivp
from scipy.stats import poisson

In [ ]:
"""
RNA velocity kinetic model (La Manno et al., Nature 2018, Fig. 1 / ED Fig. 1).

    du/dt = alpha(t) - beta(t) * u(t)          (unspliced)
    ds/dt = beta(t) * u(t) - gamma(t) * s(t)   (spliced)

Solved numerically with scipy's solve_ivp, integrating piecewise across
discontinuities in alpha(t) (the "switch-off"), and cross-checked against the
closed-form solution for piecewise-constant parameters.
"""

rng = np.random.default_rng(0)
BETA = 1.0

# ----------------------------------------------------------------------------
# Solvers
# ----------------------------------------------------------------------------
def rhs(t, y, alpha_fn, beta_fn, gamma_fn):
    u, s = y
    return [alpha_fn(t) - beta_fn(t) * u,
            beta_fn(t) * u - gamma_fn(t) * s]


def solve_switch(alpha, gamma, t_on=5.0, t_end=20.0, beta=BETA, n=2000):
    """
    Induction (alpha>0) for t < t_on, then repression (alpha=0).
    Integrates each regime separately so the solver never steps over the jump.
    """
    t = np.unique(np.concatenate([np.linspace(0, t_end, n), [t_on]]))
    segments = [(0.0, t_on, alpha, t[t <= t_on]), (t_on, t_end, 0.0, t[t > t_on])]
    y0, pieces = [0.0, 0.0], []
    for t0, t1, a, tt in segments:
        # evaluate at t1 too so we can carry the end state into the next regime
        te = np.append(tt, t1) if tt[-1] < t1 else tt
        sol = solve_ivp(rhs, (t0, t1), y0, t_eval=te, rtol=1e-9, atol=1e-11,
                        args=(lambda _t, a=a: a, lambda _t: beta, lambda _t: gamma))
        pieces.append(sol.y[:, :len(tt)])
        y0 = sol.y[:, -1]
    y = np.concatenate(pieces, axis=1)
    return t, y[0], y[1]


def analytic(t, alpha, beta, gamma, u0=0.0, s0=0.0):
    """Closed-form solution for constant alpha, beta, gamma (beta != gamma)."""
    eb, eg = np.exp(-beta * t), np.exp(-gamma * t)
    u = u0 * eb + alpha / beta * (1 - eb)
    c = (alpha - beta * u0) / (gamma - beta)
    s = s0 * eg + alpha / gamma * (1 - eg) + c * (eg - eb) * (1 if gamma != beta else np.nan)
    return u, s


def solve_generic(alpha_fn, gamma, t_end, beta=BETA, n=4000):
    t = np.linspace(0, t_end, n)
    sol = solve_ivp(rhs, (0, t_end), [0.0, 0.0], t_eval=t, rtol=1e-9, atol=1e-11,
                    max_step=0.05, args=(alpha_fn, lambda _t: beta, lambda _t: gamma))
    return t, sol.y[0], sol.y[1]


def sample_cells(t, u, s, n_cells, t_max=None):
    """Cells sampled uniformly in time, with Poisson counting noise."""
    t_max = t.max() if t_max is None else t_max
    ts = rng.uniform(0, t_max, n_cells)
    return rng.poisson(np.interp(ts, t, s)), rng.poisson(np.interp(ts, t, u))


# ----------------------------------------------------------------------------
# Cross-check numerical vs analytic (induction phase, alpha=20, gamma=0.75)
# ----------------------------------------------------------------------------
t_chk = np.linspace(0, 5, 500)
_, u_num, s_num = solve_generic(lambda _t: 20.0, 0.75, 5, n=500)
u_an, s_an = analytic(t_chk, 20, BETA, 0.75)
print(f"max |numeric - analytic|: u={np.abs(u_num-u_an).max():.2e}, "
      f"s={np.abs(s_num-s_an).max():.2e}")


def style(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(labelsize=8)


def steady_line(ax, gamma, smax, beta=BETA, color='k'):
    xs = np.array([0, smax])
    ax.plot(xs, gamma / beta * xs, color=color, linestyle='--', linewidth=0.9)


def arrow_along(ax, x, y, idx, **kw):
    ax.annotate("", xy=(x[idx + 1], y[idx + 1]), xytext=(x[idx], y[idx]),
                arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=1.3,
                                mutation_scale=12), **kw)


def panel_label(ax, lab):
    ax.text(-0.14, 1.05, lab, transform=ax.transAxes, fontsize=13,
            fontweight="bold", va="bottom")



In [ ]:
# ----------------------------------------------------------------------------
# Plot helpers
# ----------------------------------------------------------------------------
BLUE, RED, ORANGE = "#4a6fb5", "#e0262a", "#e08a3c"
psi_cmap = LinearSegmentedColormap.from_list("psi", ["#ffffff", "#f2d4e6", "#b58aa8", "#444444"])

# ---- (a) time course with Poisson observation density psi ------------------
a0, g0, t_on, t_end = 20.0, 0.75, 5.0, 18.5
t, u, s = solve_switch(a0, g0, t_on, t_end)

In [ ]:
t

In [ ]:
u

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.8))
fontsize = 14

alpha_t = np.where(t < t_on, a0, 0.0)

n_grid = np.arange(0, 36)
psi = poisson.pmf(n_grid[:, None], np.maximum(s[None, :], 1e-9))
im = ax.imshow(psi, origin="lower", aspect="auto", cmap=psi_cmap, extent=[0, t_end, -0.5, 35.5], alpha=0.55, vmin=0, vmax=1)

plt.plot([0, t_on, t_on], [a0, a0, 0], "--", color=ORANGE, lw=1, label=r"$\alpha$")
plt.plot(t, s, color=RED, lw=1.8, label="S")
plt.plot(t, u, color=BLUE, lw=1.8, label="U")
plt.xlim((0, t_end))
plt.ylim((0, 35))
plt.xlabel("t", fontsize=fontsize)
plt.ylabel("n", fontsize=fontsize)
plt.text(1.0, 29, rf"$\alpha={a0:g}$" "\n" rf"$\gamma={g0}$", fontsize=fontsize)
plt.legend(frameon=False, fontsize=fontsize, loc="upper right")
cax = ax.inset_axes([0.73, 0.25, 0.012, 0.4])
cb = fig.colorbar(im, cax=cax, ticks=[0, 0.5, 1])
cb.ax.tick_params(labelsize=fontsize)
cb.set_label(r"$\psi$", rotation=0, labelpad=-4, y=1.18, fontsize=fontsize)
style(ax); panel_label(ax, "a")


### (b) phase portrait + 500 noisy cells

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

sc, uc = sample_cells(t, u, s, 500)
ax.scatter(sc, uc, s=8, c='red', alpha=1.0, lw=0)
ax.plot(s, u, color=BLUE, lw=1.4)

steady_line(ax, g0, 28)
i_on = np.searchsorted(t, t_on)

for frac in (0.25, 0.6):                                  # induction arrows
    arrow_along(ax, s, u, int(frac * i_on))

for dt in (0.8, 3.0):                                     # repression arrows
    arrow_along(ax, s, u, np.searchsorted(t, t_on + dt))

ax.annotate(f"t = {t_on:g}", (s[i_on], u[i_on]), xytext=(4, 0),
              textcoords="offset points", fontsize=fontsize)

ax.set(xlim=(0, 30), ylim=(0, 25), xlabel="S", ylabel="U")
ax.text(1, 21, rf"$\alpha={a0:g}$" "\n" rf"$\gamma={g0}$", fontsize=fontsize)
ax.set_title("n = 500 cells sampled uniformly in time", fontsize=fontsize, style="italic")
style(ax); panel_label(ax, "b")

i_max = np.argmax(s)
ax.plot(s[i_max], u[i_max], "ko")                 # s peak = crossing point
print(t[i_max], s[i_max], u[i_max], g0 * s[i_max])  # u == γ·s here

Yes, that black dot is the crossing. It sits exactly on the dashed line because it's the maximum of s: ds/dt = 0 there, so u = γs/β. To its left the curve is above the line, which is the tail of induction plus the brief overshoot after α switches off. Below the dot, s is falling and every point is under the line, which is repression.

The gap between the "t = 5" corner and the dot is the ~0.07 time units spliced RNA takes to catch up after transcription stops. A faster decay rate γ would make it smaller, and slower splicing (smaller β) would make it bigger.

### (c) varying alpha, gamma fixed

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

for a in (10, 15, 20):
    _, uu, ss = solve_switch(a, 0.75, 5, 30)
    ax.plot(ss, uu, color=BLUE, lw=1.2)
    k = np.argmax(uu)
    ax.text(ss[k] - 3, uu[k] + 0.6, rf"$\alpha={a}$", fontsize=fontsize)

steady_line(ax, 0.8, 26)

ax.text(1, 18.5, r"$\gamma=0.75$", fontsize=fontsize)
ax.set(xlim=(0, 26), ylim=(0, 21), xlabel="s", ylabel="u")
style(ax); panel_label(ax, "c")

### (d) varying gamma, alpha fixed

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

alpha=10
for gamma, color in zip((.75, 1, 1.5), ('red', 'blue', 'green')):
    _, uu, ss = solve_switch(alpha, gamma, 5, 15)
    ax.plot(ss, uu, color=color, lw=1.2)
    k = np.argmax(uu)
    ax.text(ss[k] - 3, uu[k] + 0.6, rf"$\gamma={gamma}$", fontsize=fontsize)

    steady_line(ax, gamma, 15, color=color)

ax.text(1, 18.5, rf"$\alpha={alpha}$", fontsize=fontsize)
ax.set(xlim=(0, 15), ylim=(0, 21), xlabel="s", ylabel="u")
style(ax); panel_label(ax, "c")

### (e) oscillating transcription rate

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

g_e, period, t_end_e = 0.9, 7.0, 72
alpha_osc = lambda tt: 20 * (1 + np.sin(2 * np.pi * tt / period))
te, ue, se = solve_generic(alpha_osc, g_e, t_end_e, n=6000)

ax.plot(te, alpha_osc(te), ":", color=ORANGE, lw=1.1, label=r"$\alpha$")
ax.plot(te, se, color=RED, lw=1.2, label="S")
ax.plot(te, ue, color=BLUE, lw=1.2, label="U")
ax.set(xlim=(0, t_end_e), ylim=(0, 42), xlabel="t", ylabel="n")
plt.legend(loc='upper right', fontsize=fontsize)
plt.title(rf"$\alpha(t)$ is oscillating, no more constant")
style(ax); panel_label(ax, "e")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

g_e, period, t_end_e = 0.9, 24, 72
alpha_osc = lambda tt: 20.0 * np.where((tt % period) < period*.4, 1.0, 0.0)
te, ue, se = solve_generic(alpha_osc, g_e, t_end_e, n=6000)

ax.plot(te, alpha_osc(te), ":", color=ORANGE, lw=1.1, label=r"$\alpha$")
ax.plot(te, se, color=RED, lw=1.2, label="S")
ax.plot(te, ue, color=BLUE, lw=1.2, label="U")
ax.set(xlim=(0, t_end_e), ylim=(0, 25), xlabel="t", ylabel="n")
plt.legend(loc='upper right', fontsize=fontsize)
plt.title(rf"$\alpha(t)$ is Heaviside step function")
style(ax); panel_label(ax, "e")

### u x s - phase plane

In [ ]:
g_e, period, t_end_e = 0.9, 7.0, 72
alpha_osc = lambda tt: 20 * (1 + np.sin(2 * np.pi * tt / period))
te, ue, se = solve_generic(alpha_osc, g_e, t_end_e, n=6000)

te, ue, se, t_end_e

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

N = 5000
fontsize=14

sc, uc = sample_cells(te, ue, se, N, t_max=t_end_e)
ax.scatter(sc, uc, s=6, c='red', alpha=0.5, lw=0)
ax.plot(se, ue, color=BLUE, lw=1.1)
steady_line(ax, g_e, 38)
ax.text(2, 33, rf"$\gamma={g_e}$", fontsize=fontsize)
ax.set_xlim(0, 38)
ax.set_ylim(0, 42)
ax.set_xlabel("s", fontsize=fontsize)
ax.set_ylabel("u", fontsize=fontsize)
ax.set_title(f"n = {N} cells sampled uniformly in time", fontsize=fontsize, style="italic")
style(ax)
panel_label(ax, "f")

In [ ]:
# ----------------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------------
fig = plt.figure(figsize=(13, 6.8))
gs = fig.add_gridspec(2, 6, height_ratios=[1, 1], hspace=0.45, wspace=0.55)
ax_a = fig.add_subplot(gs[0, 0:4])
ax_b = fig.add_subplot(gs[0, 4:6])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])
ax_e1 = fig.add_subplot(gs[1, 2:4])
ax_e2 = fig.add_subplot(gs[1, 4:6])


n_grid = np.arange(0, 36)
psi = poisson.pmf(n_grid[:, None], np.maximum(s[None, :], 1e-9))
psi /= psi.max(axis=0, keepdims=True)                  # per-time normalisation
im = ax_a.imshow(psi, origin="lower", aspect="auto", cmap=psi_cmap,
                 extent=[0, t_end, -0.5, 35.5], alpha=0.55, vmin=0, vmax=1)

alpha_t = np.where(t < t_on, a0, 0.0)
ax_a.plot([0, t_on, t_on], [a0, a0, 0], "--", color=ORANGE, lw=1, label=r"$\alpha$")
ax_a.plot(t, s, color=RED, lw=1.8, label="S")
ax_a.plot(t, u, color=BLUE, lw=1.8, label="U")
ax_a.set(xlim=(0, t_end), ylim=(0, 35), xlabel="t", ylabel="n")
ax_a.text(1.0, 29, rf"$\alpha={a0:g}$" "\n" rf"$\gamma={g0}$", fontsize=9)
ax_a.legend(frameon=False, fontsize=8, loc="upper right")
cax = ax_a.inset_axes([0.73, 0.25, 0.012, 0.4])
cb = fig.colorbar(im, cax=cax, ticks=[0, 0.5, 1])
cb.ax.tick_params(labelsize=7)
cb.set_label(r"$\psi$", rotation=0, labelpad=-4, y=1.18, fontsize=10)
style(ax_a); panel_label(ax_a, "a")

# ---- (b) phase portrait + 500 noisy cells ------------------------------------
sc, uc = sample_cells(t, u, s, 500)
ax_b.scatter(sc, uc, s=3, c="0.55", alpha=0.7, lw=0)
ax_b.plot(s, u, color=BLUE, lw=1.4)
steady_line(ax_b, g0, 28)
i_on = np.searchsorted(t, t_on)
for frac in (0.25, 0.6):                                  # induction arrows
    arrow_along(ax_b, s, u, int(frac * i_on))
for dt in (0.8, 3.0):                                     # repression arrows
    arrow_along(ax_b, s, u, np.searchsorted(t, t_on + dt))
ax_b.annotate(f"t = {t_on:g}", (s[i_on], u[i_on]), xytext=(4, 0),
              textcoords="offset points", fontsize=8)
ax_b.set(xlim=(0, 30), ylim=(0, 25), xlabel="S", ylabel="U")
ax_b.text(1, 21, rf"$\alpha={a0:g}$" "\n" rf"$\gamma={g0}$", fontsize=9)
ax_b.set_title("n = 500 cells sampled uniformly in time", fontsize=8, style="italic")
style(ax_b); panel_label(ax_b, "b")

# ---- (c) varying alpha, gamma fixed -----------------------------------------
for a in (10, 15, 20):
    _, uu, ss = solve_switch(a, 0.75, 5, 30)
    ax_c.plot(ss, uu, color=BLUE, lw=1.2)
    k = np.argmax(uu)
    ax_c.text(ss[k] - 3, uu[k] + 0.6, rf"$\alpha={a}$", fontsize=8)
steady_line(ax_c, 0.75, 26)
ax_c.text(1, 18.5, r"$\gamma=0.75$", fontsize=8)
ax_c.set(xlim=(0, 26), ylim=(0, 21), xlabel="s", ylabel="u")
style(ax_c); panel_label(ax_c, "c")

# ---- (d) varying gamma ------------------------------------------------------
for a, g, pos in [(20, 2.5, (8.5, 17)), (10, 0.3, (17, 5))]:
    _, uu, ss = solve_switch(a, g, 5, 40)
    ax_d.plot(ss, uu, color=BLUE, lw=1.2)
    ax_d.text(*pos, rf"$\alpha={a}$" "\n" rf"$\gamma={g}$", fontsize=8)
ax_d.set(xlim=(0, 30), ylim=(0, 21), xlabel="s", ylabel="u")
style(ax_d); panel_label(ax_d, "d")

# ---- (e) oscillating transcription rate --------------------------------------
g_e, period, t_end_e = 0.9, 7.0, 72
alpha_osc = lambda tt: 20 * (1 + np.sin(2 * np.pi * tt / period))
te, ue, se = solve_generic(alpha_osc, g_e, t_end_e, n=6000)

ax_e1.plot(te, alpha_osc(te), ":", color=ORANGE, lw=1.1, label=r"$\alpha$")
ax_e1.plot(te, se, color=RED, lw=1.2, label="S")
ax_e1.plot(te, ue, color=BLUE, lw=1.2, label="U")
ax_e1.set(xlim=(0, t_end_e), ylim=(0, 42), xlabel="t", ylabel="n")
style(ax_e1); panel_label(ax_e1, "e")

sc, uc = sample_cells(te, ue, se, 2000, t_max=t_end_e)
ax_e2.scatter(sc, uc, s=2, c="0.55", alpha=0.5, lw=0)
ax_e2.plot(se, ue, color=BLUE, lw=1.1)
steady_line(ax_e2, g_e, 38)
ax_e2.text(2, 33, rf"$\gamma={g_e}$", fontsize=8)
ax_e2.set(xlim=(0, 38), ylim=(0, 42), xlabel="s", ylabel="u")
ax_e2.set_title("n = 2000 cells sampled uniformly in time", fontsize=8, style="italic")
style(ax_e2)

fig.savefig("rna_velocity_model.png", dpi=200, bbox_inches="tight")
print("saved rna_velocity_model.png")
